In [1]:
# ==========================================================
# Final Validation – Step 1 : Core File Check
# ==========================================================

from pathlib import Path
import pandas as pd

project_root = (
    Path.home()
    / "Desktop"
    / "Mtech_final_project"
    / "SegFusionFormer"
)

required_paths = {
    "Best U-Net":
        project_root / "models" / "best_unet.pth",

    "Best SegFusionFormer":
        project_root / "models" / "best_segfusionformer.pth",

    "NIH Metadata":
        project_root / "dataset" / "Data_Entry_2017.csv",

    "NIH Images":
        project_root / "dataset" / "images",

    "Generated Lung Masks":
        project_root / "dataset" / "lung_masks",

    "Optimal Thresholds":
        project_root / "results" / "tables" / "optimal_thresholds.csv",

    "Overall Metrics":
        project_root / "results" / "tables" / "segfusionformer_overall_metrics.csv",

    "Per-Disease Metrics":
        project_root / "results" / "tables" / "segfusionformer_per_disease_metrics_final.csv",

    "Final Package":
        project_root / "final_project_package"
}

print("=" * 80)
print("CORE PROJECT FILE VALIDATION")
print("=" * 80)

all_ok = True

for name, path in required_paths.items():

    exists = path.exists()

    print(
        f"{'✅' if exists else '❌'} "
        f"{name:30s} : {exists}"
    )

    if not exists:
        all_ok = False

print()

print(
    "Core File Status :",
    "PASS ✅" if all_ok else "CHECK REQUIRED ⚠️"
)

CORE PROJECT FILE VALIDATION
✅ Best U-Net                     : True
✅ Best SegFusionFormer           : True
✅ NIH Metadata                   : True
✅ NIH Images                     : True
✅ Generated Lung Masks           : True
✅ Optimal Thresholds             : True
✅ Overall Metrics                : True
✅ Per-Disease Metrics            : True
✅ Final Package                  : True

Core File Status : PASS ✅


In [2]:
# ==========================================================
# Final Validation – Step 2 : Dataset Integrity
# ==========================================================

metadata_path = (
    project_root
    / "dataset"
    / "Data_Entry_2017.csv"
)

images_dir = (
    project_root
    / "dataset"
    / "images"
)

masks_dir = (
    project_root
    / "dataset"
    / "lung_masks"
)

df = pd.read_csv(metadata_path)

image_count = len(
    list(images_dir.glob("*.png"))
)

mask_count = len(
    list(masks_dir.glob("*_mask.png"))
)

print("=" * 80)
print("DATASET INTEGRITY")
print("=" * 80)

print("Metadata Rows :", len(df))
print("Images        :", image_count)
print("Lung Masks    :", mask_count)

print()

if (
    len(df) == 112120
    and image_count == 112120
    and mask_count == 112120
):

    print("✅ Dataset integrity check passed.")

else:

    print("⚠️ Dataset counts require inspection.")

DATASET INTEGRITY
Metadata Rows : 112120
Images        : 112120
Lung Masks    : 112120

✅ Dataset integrity check passed.


In [3]:
# ==========================================================
# Final Validation – Step 3 : Checkpoint Inspection
# ==========================================================

import torch

unet_path = (
    project_root
    / "models"
    / "best_unet.pth"
)

classifier_path = (
    project_root
    / "models"
    / "best_segfusionformer.pth"
)

unet_checkpoint = torch.load(
    unet_path,
    map_location="cpu"
)

classifier_checkpoint = torch.load(
    classifier_path,
    map_location="cpu"
)

print("=" * 80)
print("CHECKPOINT VALIDATION")
print("=" * 80)

print("\nU-Net checkpoint:")
print(type(unet_checkpoint))

if isinstance(unet_checkpoint, dict):
    print("Keys:", list(unet_checkpoint.keys()))

    print(
        "Validation Dice:",
        unet_checkpoint.get(
            "val_dice",
            "Not stored"
        )
    )

print("\nSegFusionFormer checkpoint:")
print(type(classifier_checkpoint))

if isinstance(classifier_checkpoint, dict):
    print("Keys:", list(classifier_checkpoint.keys()))

CHECKPOINT VALIDATION

U-Net checkpoint:
<class 'dict'>
Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_dice', 'val_loss']
Validation Dice: 0.9600503588544911

SegFusionFormer checkpoint:
<class 'collections.OrderedDict'>
Keys: ['mask_attention.alpha', 'mask_attention.attention_generator.0.weight', 'mask_attention.attention_generator.0.bias', 'mask_attention.attention_generator.1.weight', 'mask_attention.attention_generator.1.bias', 'mask_attention.attention_generator.1.running_mean', 'mask_attention.attention_generator.1.running_var', 'mask_attention.attention_generator.1.num_batches_tracked', 'mask_attention.attention_generator.3.weight', 'mask_attention.attention_generator.3.bias', 'mask_attention.attention_generator.5.weight', 'mask_attention.attention_generator.5.bias', 'cnn_branch.features.0.0.weight', 'cnn_branch.features.0.1.weight', 'cnn_branch.features.0.1.bias', 'cnn_branch.features.0.1.running_mean', 'cnn_branch.features.0.1.running_var', 'cnn_branch.featur

In [4]:
# ==========================================================
# Final Validation – Step 4 : Results Table Validation
# ==========================================================

tables_dir = (
    project_root
    / "results"
    / "tables"
)

threshold_df = pd.read_csv(
    tables_dir / "optimal_thresholds.csv"
)

overall_df = pd.read_csv(
    tables_dir / "segfusionformer_overall_metrics.csv"
)

per_class_df = pd.read_csv(
    tables_dir / "segfusionformer_per_disease_metrics_final.csv"
)

print("=" * 80)
print("RESULT TABLE VALIDATION")
print("=" * 80)

print(
    "Threshold Classes :",
    len(threshold_df)
)

print(
    "Per-Disease Rows  :",
    len(per_class_df)
)

print(
    "Overall Metrics   :",
    len(overall_df)
)

assert len(threshold_df) == 14
assert len(per_class_df) == 14

print(
    "\n✅ Disease-class tables contain all 14 classes."
)

RESULT TABLE VALIDATION
Threshold Classes : 14
Per-Disease Rows  : 14
Overall Metrics   : 9

✅ Disease-class tables contain all 14 classes.


# Checkpoint 3 – Model Validation

In [5]:
# ==========================================================
# Checkpoint 3 – Step 3.1 : Imports
# ==========================================================

from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import timm

In [6]:
# ==========================================================
# Checkpoint 3 – Step 3.2 : Model Paths
# ==========================================================

project_root = (
    Path.home()
    / "Desktop"
    / "Mtech_final_project"
    / "SegFusionFormer"
)

models_dir = project_root / "models"

unet_path = models_dir / "best_unet.pth"

classifier_path = models_dir / "best_segfusionformer.pth"

print(unet_path)
print(classifier_path)

/Users/solomonrajan/Desktop/Mtech_final_project/SegFusionFormer/models/best_unet.pth
/Users/solomonrajan/Desktop/Mtech_final_project/SegFusionFormer/models/best_segfusionformer.pth


In [7]:
# ==========================================================
# Checkpoint 3 – Step 3.3 : Verify Files
# ==========================================================

print("=" * 70)
print("MODEL FILE VALIDATION")
print("=" * 70)

print("U-Net Exists          :", unet_path.exists())
print("SegFusionFormer Exists:", classifier_path.exists())

MODEL FILE VALIDATION
U-Net Exists          : True
SegFusionFormer Exists: True


In [8]:
# ==========================================================
# Checkpoint 3 – Step 3.4 : Load Checkpoints
# ==========================================================

device = torch.device("cpu")

unet_checkpoint = torch.load(
    unet_path,
    map_location=device
)

classifier_checkpoint = torch.load(
    classifier_path,
    map_location=device
)

print("=" * 70)
print("CHECKPOINTS LOADED")
print("=" * 70)

print("U-Net Type           :", type(unet_checkpoint))
print("SegFusionFormer Type :", type(classifier_checkpoint))

CHECKPOINTS LOADED
U-Net Type           : <class 'dict'>
SegFusionFormer Type : <class 'collections.OrderedDict'>


In [9]:
# ==========================================================
# Checkpoint 3 – Step 3.5 : Check Contents
# ==========================================================

if isinstance(unet_checkpoint, dict):

    print("\nU-Net Keys:")
    print(list(unet_checkpoint.keys()))

if isinstance(classifier_checkpoint, dict):

    print("\nSegFusionFormer Keys:")
    print(list(classifier_checkpoint.keys()))


U-Net Keys:
['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_dice', 'val_loss']

SegFusionFormer Keys:
['mask_attention.alpha', 'mask_attention.attention_generator.0.weight', 'mask_attention.attention_generator.0.bias', 'mask_attention.attention_generator.1.weight', 'mask_attention.attention_generator.1.bias', 'mask_attention.attention_generator.1.running_mean', 'mask_attention.attention_generator.1.running_var', 'mask_attention.attention_generator.1.num_batches_tracked', 'mask_attention.attention_generator.3.weight', 'mask_attention.attention_generator.3.bias', 'mask_attention.attention_generator.5.weight', 'mask_attention.attention_generator.5.bias', 'cnn_branch.features.0.0.weight', 'cnn_branch.features.0.1.weight', 'cnn_branch.features.0.1.bias', 'cnn_branch.features.0.1.running_mean', 'cnn_branch.features.0.1.running_var', 'cnn_branch.features.0.1.num_batches_tracked', 'cnn_branch.features.1.0.block.0.0.weight', 'cnn_branch.features.1.0.block.0.1.weight', 'cnn_branch.fe

In [10]:
# ==========================================================
# Checkpoint 3 – Step 3.6 : Saved Metrics
# ==========================================================

print("=" * 70)
print("SAVED METRICS")
print("=" * 70)

if isinstance(unet_checkpoint, dict):

    print(
        "Validation Dice :",
        unet_checkpoint.get(
            "val_dice",
            "Not stored"
        )
    )

    print(
        "Validation Loss :",
        unet_checkpoint.get(
            "val_loss",
            "Not stored"
        )
    )

print()

if isinstance(classifier_checkpoint, dict):

    print(
        "Best AUROC :",
        classifier_checkpoint.get(
            "best_auroc",
            classifier_checkpoint.get(
                "val_auroc",
                "Not stored"
            )
        )
    )

    print(
        "Best Epoch :",
        classifier_checkpoint.get(
            "best_epoch",
            classifier_checkpoint.get(
                "epoch",
                "Not stored"
            )
        )
    )

SAVED METRICS
Validation Dice : 0.9600503588544911
Validation Loss : 0.10720872776261692

Best AUROC : Not stored
Best Epoch : Not stored


In [11]:
# ==========================================================
# Checkpoint 3 – Step 3.7 : Final Validation
# ==========================================================

print("=" * 80)
print("MODEL VALIDATION SUMMARY")
print("=" * 80)

status = (
    unet_path.exists()
    and
    classifier_path.exists()
)

print(
    "Checkpoint Files :",
    "PASS ✅" if status else "FAIL ❌"
)

if status:

    print("\nModels are available and checkpoints load successfully.")

else:

    print("\nOne or more checkpoint files are missing.")

MODEL VALIDATION SUMMARY
Checkpoint Files : PASS ✅

Models are available and checkpoints load successfully.


In [12]:
# ==========================================================
# Checkpoint 4 – Step 4.1 : Inference Paths
# ==========================================================

images_dir = (
    project_root
    / "dataset"
    / "images"
)

results_dir = (
    project_root
    / "results"
)

demo_figures_dir = (
    results_dir
    / "demo"
    / "figures"
)

demo_reports_dir = (
    results_dir
    / "demo"
    / "reports"
)

demo_figures_dir.mkdir(
    parents=True,
    exist_ok=True
)

demo_reports_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("✅ Inference paths ready.")

✅ Inference paths ready.


In [13]:
# ==========================================================
# Checkpoint 4 – Step 4.2 : Select Test X-ray
# ==========================================================

test_images = sorted(
    images_dir.glob("*.png")
)

if len(test_images) == 0:
    raise RuntimeError(
        "No NIH chest X-ray images were found."
    )

validation_image_path = test_images[0]

print("=" * 70)
print("END-TO-END TEST IMAGE")
print("=" * 70)

print(validation_image_path)

print(
    "\nExists:",
    validation_image_path.exists()
)

END-TO-END TEST IMAGE
/Users/solomonrajan/Desktop/Mtech_final_project/SegFusionFormer/dataset/images/00000001_000.png

Exists: True
